# AVI vs ClinVar: independent evaluation (Kaggle runner)

Research use only. No clinical or diagnostic claims; not medical advice.
Pre-registered design: `design.md` in the repository. **Keep this notebook PRIVATE**: it will hold AlphaGenome outputs in its working folder.

**Before you run (one-time setup)**
1. Right sidebar → **Session options** → **Internet: On** (needs phone-verified Kaggle account).
2. Top menu **Add-ons → Secrets → Add secret**: Label `ALPHAGENOME_API_KEY`, Value = your personal key. Tick the toggle so this notebook can use it.
3. Check `REPO_URL` in the first code cell (already set to the GitHub repo).
4. Keep `DRY_RUN = True` the first time. Run all cells. Read the dry-run output. Then set `DRY_RUN = False` and run again.

In [ ]:
DRY_RUN = True          # True: 20 variants only. Set to False for the full run (after checking the dry run).
REPO_URL = "https://github.com/Arths17/avi-clinvar-eval.git"   # <-- edit this

In [ ]:
import os, subprocess, sys
assert "<YOUR-USERNAME>" not in REPO_URL, "Edit REPO_URL in the cell above first."
os.chdir("/kaggle/working")
if not os.path.isdir("avi-clinvar-eval"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "avi-clinvar-eval"], check=True)
os.chdir("/kaggle/working/avi-clinvar-eval")
req = "requirements-py311.txt" if sys.version_info[:2] == (3, 11) else "requirements.txt"
print("Python", sys.version.split()[0], "-> installing from", req)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
# Load the API key from Kaggle Secrets into an environment variable. The key is never printed.
from kaggle_secrets import UserSecretsClient
os.environ["ALPHAGENOME_API_KEY"] = UserSecretsClient().get_secret("ALPHAGENOME_API_KEY")
print("API key loaded:", bool(os.environ.get("ALPHAGENOME_API_KEY")))

### Step 1: download ClinVar (public NCBI data, ~189 MB) and prepare the samples (no API calls)

In [ ]:
!python -m avi_eval download --yes
!python -m avi_eval prepare

**Gate:** open `data/mc_term_counts.csv` and `results/attrition.csv` (left file browser, or run the next cell) and send them to Claude *before* scoring. Any consequence term shown as `UNMAPPED` must be discussed first.

In [ ]:
import pandas as pd
print(pd.read_csv("results/attrition.csv").to_string(index=False))
print(pd.read_csv("data/mc_term_counts.csv").head(30).to_string(index=False))

### Step 2: dry run (20 variants) or full run

In [ ]:
if DRY_RUN:
    !python -m avi_eval score --dry-run
else:
    !python -m avi_eval score --cohort all

### Step 3: analysis (only after the full run)

In [ ]:
if not DRY_RUN:
    !python -m avi_eval analyze --n-boot 2000
    print(pd.read_csv("results/metrics.csv").round(3).to_string(index=False))
else:
    print("Dry run only: no analysis. Check the dry-run table above, then set DRY_RUN = False.")

### Step 4: bundle the shareable outputs (no per-variant AlphaGenome scores)

In [ ]:
import shutil, glob
if not DRY_RUN:
    os.makedirs("share", exist_ok=True)
    for pat in ["results/*.csv", "results/*.json", "results/figures/*.png", "data/sample_*.csv", "data/mc_term_counts.csv"]:
        for f in glob.glob(pat):
            dest = os.path.join("share", f)
            os.makedirs(os.path.dirname(dest), exist_ok=True)
            shutil.copy(f, dest)
    shutil.make_archive("/kaggle/working/results_bundle", "zip", "share")
    print("Download /kaggle/working/results_bundle.zip from the Output tab.")